In [2]:
!pip install faker


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import random
import sqlite3
import re

from faker import Faker
from datetime import datetime, timedelta

In [4]:
fake = Faker()

random.seed(42)
np.random.seed(42)
Faker.seed(42)

In [5]:
NUM_CUSTOMERS = 500
NUM_PRODUCTS = 500
NUM_ORDERS = 500
NUM_ORDER_ITEMS = 500

PHASE 1: Python & SQL (Local Environment)

Part 1: Data Generation

#  customers.csv

In [6]:
customers = []

In [7]:
customer_types = ["REGULAR", "PREMIUM", "VIP"]

In [8]:
for i in range(1, NUM_CUSTOMERS + 1):

    customer_id = f"CUST{i:03d}"

    customer_name = fake.name()

    registration_date = fake.date_between(
        start_date="-5y",
        end_date="today"
    )
    customer_type = random.choice(customer_types)
    email = fake.email()
    if random.random() < 0.02:
        if random.choice([True, False]):
            email = email.replace("@", "")
        else:
            email = email.split("@")[0] + "@"
    customers.append({
        "customer_id": customer_id,
        "customer_name": customer_name,
        "email": email,
        "registration_date": registration_date,
        "customer_type": customer_type
    })



In [9]:
customers_df = pd.DataFrame(customers)

In [10]:
customers_df.head()

,customer_id,customer_name,email,registration_date,customer_type
0,CUST001,Allison Hill,jillrhodes@example.net,2023-06-04,VIP
1,CUST002,Javier Johnson,williamjohnson@example.org,2023-05-21,VIP
2,CUST003,Meredith Barnes,lisa02@example.net,2025-05-06,REGULAR
3,CUST004,Donald Lewis,daviscolin@example.com,2024-05-21,REGULAR
4,CUST005,Renee Blair,dudleynicholas@example.net,2026-02-01,VIP


In [11]:
customers_df.to_csv("customers.csv", index=False)
print("customers.csv created")

customers.csv created


# products.csv 

In [12]:
products = []

In [13]:
categories = {
    "Electronics": ["Mobile", "Laptop", "Headphones", "Camera"],
    "Clothing": ["Shirt", "Jeans", "Shoes", "Jacket"],
    "Home": ["Chair", "Table", "Sofa", "Lamp"],
    "Books": ["Fiction", "Science", "History", "Comics"]
}

In [14]:
for i in range(1, NUM_PRODUCTS + 1):
    product_id = f"PROD{i:03d}"
    category = random.choice(list(categories.keys()))
    subcategory = random.choice(categories[category])
    product_name = fake.word().title() + " " + subcategory
    if random.random() < 0.05:
        issue = random.choice(["spaces", "uppercase", "lowercase", "mixed"])

        if issue == "spaces":
            product_name = "  " + product_name + "  "

        elif issue == "uppercase":
            product_name = product_name.upper()

        elif issue == "lowercase":
            product_name = product_name.lower()

        elif issue == "mixed":
            product_name = product_name.swapcase()

    cost_price = round(random.uniform(100, 5000), 2)

    products.append({
        "product_id": product_id,
        "product_name": product_name,
        "category": category,
        "subcategory": subcategory,
        "cost_price": cost_price
    })

products_df = pd.DataFrame(products)

In [15]:
products_df.to_csv("products.csv", index=False)

print("products.csv created")

products.csv created


In [16]:
products_df.sample(10)

,product_id,product_name,category,subcategory,cost_price
361,PROD362,Market Mobile,Electronics,Mobile,2602.31
73,PROD074,Main Comics,Books,Comics,1156.29
374,PROD375,Court Shoes,Clothing,Shoes,4062.78
155,PROD156,Various Jacket,Clothing,Jacket,307.54
104,PROD105,Follow Chair,Home,Chair,2834.02
394,PROD395,Degree Shoes,Clothing,Shoes,257.93
377,PROD378,Much History,Books,History,3095.41
124,PROD125,Factor Headphones,Electronics,Headphones,885.53
68,PROD069,Arm Jacket,Clothing,Jacket,3730.75
450,PROD451,Site Chair,Home,Chair,243.85


# orders.csv

In [17]:
orders = []

In [18]:
order_status = [
    "PLACED",
    "SHIPPED",
    "DELIVERED",
    "CANCELLED",
    "RETURNED"
]

In [19]:
region_codes = [
    "NORTH",
    "SOUTH",
    "EAST",
    "WEST",
    "CENTRAL"
]

In [20]:
customer_ids = customers_df["customer_id"].tolist()

In [21]:
for i in range(1, NUM_ORDERS + 1):

    order_id = f"ORD{i:03d}"

    # 5% NULL customer IDs
    if random.random() < 0.05:
        customer_id = None
    else:
        customer_id = random.choice(customer_ids)

    # Random order date in last 2 years
    order_datetime = fake.date_time_between(
        start_date="-2y",
        end_date="now"
    )

    # Correct format
    order_date = order_datetime.strftime("%Y-%m-%d %H:%M:%S")

    # Introduce some wrong date formats
    if random.random() < 0.05:
        order_date = order_datetime.strftime("%d-%m-%Y")

    status = random.choice(order_status)

    region_code = random.choice(region_codes)

    orders.append({
        "order_id": order_id,
        "customer_id": customer_id,
        "order_date": order_date,
        "status": status,
        "region_code": region_code
    })

In [22]:
orders_df = pd.DataFrame(orders)

In [23]:
orders_df.sample(10)

,order_id,customer_id,order_date,status,region_code
306,ORD307,CUST460,2025-03-14 04:47:45,PLACED,WEST
449,ORD450,CUST103,2025-10-30 14:55:05,RETURNED,EAST
123,ORD124,CUST061,2025-08-18 04:03:12,PLACED,WEST
194,ORD195,CUST397,2025-12-19 19:12:08,RETURNED,EAST
66,ORD067,CUST310,2024-12-31 06:43:37,SHIPPED,WEST
394,ORD395,CUST318,2026-04-24 16:11:46,RETURNED,SOUTH
430,ORD431,CUST258,2026-03-05 18:54:42,SHIPPED,NORTH
345,ORD346,CUST121,2025-10-02 08:02:54,PLACED,SOUTH
498,ORD499,CUST092,2024-09-24 05:52:49,PLACED,WEST
237,ORD238,CUST435,2025-01-16 00:53:25,PLACED,CENTRAL


In [24]:
order_ids = orders_df["order_id"].tolist()
product_ids = products_df["product_id"].tolist()



#  order_items.csv

In [25]:
order_items = []

In [26]:
for i in range(1, NUM_ORDER_ITEMS + 1):

    item_id = f"ITEM{i:03d}"

    order_id = random.choice(order_ids)

    product_id = random.choice(product_ids)

    quantity = random.randint(1, 5)
    if random.random() < 0.03:
        quantity = -quantity

    unit_price = round(random.uniform(100, 5000), 2)

    discount_percent = round(random.uniform(0, 100), 2)

    order_items.append({
        "item_id": item_id,
        "order_id": order_id,
        "product_id": product_id,
        "quantity": quantity,
        "unit_price": unit_price,
        "discount_percent": discount_percent
    })

In [27]:

order_items_df = pd.DataFrame(order_items)

In [28]:
print("Customers:", len(customers_df))
print("Products:", len(products_df))
print("Orders:", len(orders_df))
print("Order Items:", len(order_items_df))

Customers: 500
Products: 500
Orders: 500
Order Items: 500


In [29]:
print("NULL Customer IDs:", orders_df["customer_id"].isnull().sum())

NULL Customer IDs: 22


In [30]:
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

invalid_emails = customers_df[
    customers_df["email"].str.match(email_pattern) == False
]

print("Invalid Emails:", len(invalid_emails))
invalid_emails.head()

Invalid Emails: 6


,customer_id,customer_name,email,registration_date,customer_type
66,CUST067,Susan Vargas,dayderek@,2021-10-03,PREMIUM
148,CUST149,Brittney Olson,megan51example.com,2022-06-12,REGULAR
169,CUST170,Lisa Cox,vortizexample.net,2023-12-06,REGULAR
173,CUST174,Theresa Osborn,lucasrodneyexample.com,2026-01-01,VIP
457,CUST458,Michael Brown,perrybrittany@,2022-11-18,VIP


In [31]:
negative_qty = order_items_df[order_items_df["quantity"] < 0]

print("Negative Quantities:", len(negative_qty))
negative_qty.head()

Negative Quantities: 23


,item_id,order_id,product_id,quantity,unit_price,discount_percent
17,ITEM018,ORD100,PROD001,-1,3164.76,11.79
19,ITEM020,ORD287,PROD413,-2,1085.82,28.16
26,ITEM027,ORD162,PROD285,-5,2929.25,33.17
36,ITEM037,ORD443,PROD268,-1,4668.45,68.71
38,ITEM039,ORD393,PROD338,-3,3816.62,63.89


In [32]:
wrong_dates = orders_df[
    orders_df["order_date"].str.match(r"\d{2}-\d{2}-\d{4}")
]

print("Wrong Date Format:", len(wrong_dates))
wrong_dates.head()

Wrong Date Format: 25


,order_id,customer_id,order_date,status,region_code
8,ORD009,CUST252,17-12-2024,PLACED,SOUTH
55,ORD056,CUST448,28-03-2026,RETURNED,CENTRAL
76,ORD077,NaN,03-12-2025,RETURNED,SOUTH
78,ORD079,CUST263,19-02-2025,PLACED,WEST
111,ORD112,CUST422,12-07-2024,DELIVERED,WEST


In [33]:
messy_products = products_df[
    (products_df["product_name"] != products_df["product_name"].str.strip()) |
    (products_df["product_name"] != products_df["product_name"].str.title())
]

print("Messy Product Names:", len(messy_products))
messy_products.head()

Messy Product Names: 27


,product_id,product_name,category,subcategory,cost_price
13,PROD014,oFFER sOFA,Home,Sofa,1444.95
44,PROD045,SITE LAPTOP,Electronics,Laptop,2985.38
75,PROD076,oFTEN hISTORY,Books,History,4923.78
80,PROD081,Enough Camera,Electronics,Camera,4324.34
82,PROD083,decade camera,Electronics,Camera,942.18


In [34]:
orders_df.to_csv("orders.csv", index=False)
order_items_df.to_csv("order_items.csv", index=False)


# Phase 2: Data Cleaning

1. clean_orders() - Fix date formats, handle NULL customer_ids 

In [35]:
def clean_orders(df):

    cleaned_order_df = df.copy()
    cleaned_order_df["order_date"] = pd.to_datetime(
        cleaned_order_df["order_date"],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )
    cleaned_order_df["customer_id"] = cleaned_order_df["customer_id"].fillna("UNKNOWN")

    return cleaned_order_df

In [36]:
cleaned_order_df = clean_orders(orders_df)

In [37]:
cleaned_order_df.to_csv("orders_cleaned.csv", index=False)

2. clean_products() - Normalize product names (trim spaces, title case) 


In [38]:
def clean_products(df):
  cleaned_product_df=df.copy()
  cleaned_product_df["product_name"]=cleaned_product_df["product_name"].str.strip()
  cleaned_product_df["product_name"]=cleaned_product_df["product_name"].str.title()
  return cleaned_product_df

In [39]:
cleaned_products_df = clean_products(products_df)

3. validate_emails() - Return list of customer_ids with invalid emails 

In [40]:
def validate_emails(df):
  clean_customers_df=df.loc[
      ~df["email"].str.contains(
          r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
          regex=True,
          na=False
      ),"customer_id"
  ].tolist()
  return clean_customers_df

In [41]:
invalid_customer_ids = validate_emails(customers_df)

print("Invalid Customer IDs:")
print(invalid_customer_ids)

Invalid Customer IDs:
['CUST067', 'CUST149', 'CUST170', 'CUST174', 'CUST458', 'CUST493']


In [42]:
clean_customers_df = customers_df[
    customers_df["email"].str.contains(
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        regex=True,
        na=False
    )
]

4. check_referential_integrity() - Find order_items that reference non-existent orders

In [43]:
ref_order_items=order_items_df.copy()

In [44]:
ref_order_items.loc[0,"order_id"]="ORD999"

In [45]:
def check_referential_integrity(ref_order_items, orders_df):

    invalid_orders = order_items_df[
        ~ref_order_items["order_id"].isin(orders_df["order_id"])
    ]

    return invalid_orders

In [46]:
check_referential_integrity(ref_order_items, orders_df)

,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,ITEM001,ORD321,PROD093,1,4806.13,0.35


In [47]:

cleaned_products_df.to_csv("products_cleaned.csv", index=False)

clean_customers_df.to_csv("customers_cleaned.csv", index=False)
order_items_df.to_csv("order_items_cleaned.csv", index=False)

print("All cleaned CSV files saved successfully!")

All cleaned CSV files saved successfully!


 a report of all issues found

In [48]:
issues_report = {
    "Total Customers": len(customers_df),
    "Total Products": len(products_df),
    "Total Orders": len(orders_df),
    "Total Order Items": len(order_items_df),

    "Invalid Emails": len(invalid_customer_ids),
    "NULL Customer IDs": orders_df["customer_id"].isnull().sum(),
    "Wrong Date Formats": len(wrong_dates),
    "Messy Product Names": len(messy_products),
    "Negative Quantities": len(negative_qty),
    "Invalid Order References": len(
        check_referential_integrity(ref_order_items, orders_df)
    )
}

In [49]:
print("=" * 40)
print("DATA QUALITY REPORT")
print("=" * 40)

for key, value in issues_report.items():
    print(f"{key}: {value}")

DATA QUALITY REPORT
Total Customers: 500
Total Products: 500
Total Orders: 500
Total Order Items: 500
Invalid Emails: 6
NULL Customer IDs: 22
Wrong Date Formats: 25
Messy Product Names: 27
Negative Quantities: 23
Invalid Order References: 1


In [50]:
with open("issue_report.txt", "w") as file:
    file.write("DATA QUALITY REPORT\n")
    file.write("=" * 40 + "\n")

    for key, value in issues_report.items():
        file.write(f"{key}: {value}\n")

print("Issue report saved ")

Issue report saved 


## Part 3: SQL Analysis

In [51]:
import sqlite3
import pandas as pd

In [52]:
conn = sqlite3.connect("ecommerce.db")

print("Database created successfully!")

Database created successfully!


In [53]:
customers = pd.read_csv("customers_cleaned.csv")
products = pd.read_csv("products_cleaned.csv")
orders = pd.read_csv("orders_cleaned.csv")
order_items = pd.read_csv("order_items_cleaned.csv")

print("All cleaned CSV files loaded successfully!")


All cleaned CSV files loaded successfully!


In [54]:
customers.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

products.to_sql(
    "products",
    conn,
    if_exists="replace",
    index=False
)

orders.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

order_items.to_sql(
    "order_items",
    conn,
    if_exists="replace",
    index=False
)

print("Tables created successfully!")

Tables created successfully!


In [55]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tables = pd.read_sql(query, conn)

tables

,name
0,customers
1,products
2,orders
3,order_items


In [56]:
pd.read_sql(
    "SELECT * FROM order_items LIMIT 1;",
    conn
)


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,ITEM001,ORD321,PROD093,1,4806.13,0.35


In [57]:
pd.read_sql(
    "SELECT * FROM products LIMIT 1;",
    conn
)


,product_id,product_name,category,subcategory,cost_price
0,PROD001,Term Fiction,Books,Fiction,1250.37


# Basic Queries

1. Total revenue per category (revenue = quantity × unit_price × (1 - discount_percent/100))

In [58]:
query1 = """
SELECT
    p.category,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 2) AS total_revenue
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

revenue_per_category = pd.read_sql(query1, conn)
revenue_per_category

,category,total_revenue
0,Clothing,530195.24
1,Books,503257.34
2,Electronics,384507.93
3,Home,357150.58


2. Top 10 customers by total order value

In [59]:
query2 = """
SELECT
    o.customer_id,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 2) AS total_value
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.customer_id != 'UNKNOWN'
GROUP BY o.customer_id
ORDER BY total_value DESC
LIMIT 10;"""
top_customers = pd.read_sql(query2, conn)

top_customers

,customer_id,total_value
0,CUST079,43148.87
1,CUST021,40435.33
2,CUST340,28429.96
3,CUST422,27843.18
4,CUST121,26699.83
5,CUST101,26608.12
6,CUST333,25003.91
7,CUST167,24375.20
8,CUST295,24160.47
9,CUST325,23001.94


Query 3: Month-wise Order Count for the Last 12 Months

In [60]:
query3 = """
SELECT
    strftime('%Y-%m', order_date) AS month,
    COUNT(order_id) AS total_orders
FROM orders
WHERE order_date >= date('now', '-12 months')
GROUP BY month
ORDER BY month;
"""

monthly_orders = pd.read_sql(query3, conn)

monthly_orders

,month,total_orders
0,2025-07,17
1,2025-08,14
2,2025-09,19
3,2025-10,27
4,2025-11,17
5,2025-12,27
6,2026-01,10
7,2026-02,12
8,2026-03,19
9,2026-04,15


Query 4: Find customers who placed orders but never had any item delivered

In [61]:
query4 = """
SELECT DISTINCT o.customer_id
FROM orders o
WHERE o.customer_id <> 'UNKNOWN'
AND o.customer_id NOT IN (
    SELECT customer_id
    FROM orders
    WHERE status = 'DELIVERED'
);
"""

not_delivered = pd.read_sql(query4, conn)

not_delivered

,customer_id
0,CUST242
1,CUST406
2,CUST409
3,CUST486
4,CUST252
...,...
198,CUST203
199,CUST238
200,CUST197
201,CUST290


Query 5: Products that were ordered but had more returns than purchases

In [62]:
query5 = """SELECT
    p.product_id,
    p.product_name,
    SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS total_purchased,
    SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) AS total_returned
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.product_id, p.product_name
HAVING total_returned > total_purchased;
"""

more_returns = pd.read_sql(query5, conn)

more_returns

,product_id,product_name,total_purchased,total_returned
0,PROD001,Term Fiction,0,1
1,PROD099,Friend Shoes,0,1
2,PROD249,Range Sofa,0,4
3,PROD277,Dog Camera,0,2
4,PROD285,Structure Comics,2,5
5,PROD293,Less Chair,0,5
6,PROD326,Age History,0,2
7,PROD338,Page Lamp,0,3
8,PROD364,Meet Shoes,0,1
9,PROD415,Most Mobile,0,5


Query 6: Return Rate per Category

In [63]:
query6 = """SELECT
    p.category,
    ROUND(
        SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) * 1.0
        / SUM(ABS(oi.quantity)), 4
    ) AS return_rate
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY return_rate DESC;
"""

return_rate = pd.read_sql(query6, conn)

return_rate

,category,return_rate
0,Home,0.0574
1,Electronics,0.0338
2,Clothing,0.0334
3,Books,0.0284


Query 7: Running Totals with Window Functions

In [64]:
query7 = """
WITH daily_sales AS (
SELECT o.region_code,  DATE(o.order_date) AS order_date,
ROUND(SUM(oi.quantity * oi.unit_price *(1 - oi.discount_percent / 100.0)),
2) AS daily_revenue
FROM orders o
JOIN order_items oi
 ON o.order_id = oi.order_id
GROUP BY o.region_code,DATE(o.order_date)
)

SELECT

    region_code,
    order_date,
    daily_revenue,

    SUM(daily_revenue) OVER(
        PARTITION BY region_code
        ORDER BY order_date
    ) AS running_total

FROM daily_sales

ORDER BY
    region_code,
    order_date;
"""

running_total = pd.read_sql(query7, conn)

running_total

,region_code,order_date,daily_revenue,running_total
0,CENTRAL,2024-01-11,4582.06,4582.06
1,CENTRAL,2024-02-11,-609.93,3972.13
2,CENTRAL,2024-06-09,11078.78,15050.91
3,CENTRAL,2024-07-18,14539.42,29590.33
4,CENTRAL,2024-08-23,-1560.11,28030.22
...,...,...,...,...
300,WEST,2026-05-03,1302.77,273141.60
301,WEST,2026-05-16,3733.74,276875.34
302,WEST,2026-06-04,2827.15,279702.49
303,WEST,2026-12-02,3659.72,283362.21


Query 8)Ranking with DENSE_RANK

In [65]:
query8 = """WITH product_revenue AS (
    SELECT
        p.category,
        p.product_name,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS total_revenue
    FROM order_items oi
    JOIN products p ON p.product_id = oi.product_id
    GROUP BY p.category, p.product_name
)
SELECT
    category,
    product_name,
    ROUND(total_revenue, 2) AS total_revenue,
    DENSE_RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
FROM product_revenue
ORDER BY category, rank_in_category;
"""

product_rank = pd.read_sql(query8, conn)

product_rank

,category,product_name,total_revenue,rank_in_category
0,Books,Easy History,24162.40,1
1,Books,Drug Comics,22592.60,2
2,Books,Question Fiction,22095.45,3
3,Books,Action Science,21416.31,4
4,Books,Respond Fiction,20982.62,5
...,...,...,...,...
302,Home,Note Chair,24.48,60
303,Home,Score Chair,-541.88,61
304,Home,Less Chair,-2657.07,62
305,Home,Page Lamp,-4134.54,63


Query 9: LAG Analysis

In [66]:
query9 = """
WITH customer_orders AS (
SELECT customer_id,DATE(order_date) AS order_date,
LAG(DATE(order_date))
OVER(
PARTITION BY customer_id
ORDER BY DATE(order_date)
) AS previous_order_date
FROM orders
WHERE customer_id<>'UNKNOWN'
),
gap_days AS(
SELECT
customer_id,
order_date,
previous_order_date,
ROUND(
julianday(order_date)-
julianday(previous_order_date)
,2)
AS days_gap
FROM customer_orders
)
SELECT
customer_id,
order_date,
previous_order_date,
days_gap,
CASE
WHEN AVG(days_gap)
OVER(PARTITION BY customer_id)>30
THEN 'At Risk'
ELSE 'Active'

END AS customer_status

FROM gap_days

ORDER BY

customer_id,

order_date;
"""

customer_gap = pd.read_sql(query9, conn)

customer_gap

,customer_id,order_date,previous_order_date,days_gap,customer_status
0,CUST004,2026-03-25,NaN,NaN,At Risk
1,CUST004,2026-10-04,2026-03-25,193.0,At Risk
2,CUST007,2024-03-09,NaN,NaN,Active
3,CUST010,2025-04-04,NaN,NaN,Active
4,CUST011,2025-05-28,NaN,NaN,At Risk
...,...,...,...,...,...
473,CUST494,2026-05-03,2024-06-09,693.0,At Risk
474,CUST497,2024-08-18,NaN,NaN,At Risk
475,CUST497,2024-11-24,2024-08-18,98.0,At Risk
476,CUST499,2025-01-02,NaN,NaN,Active


10. CTE with Multiple Levels

In [67]:
query10 = """WITH monthly_revenue AS (
    SELECT
        o.customer_id,
        strftime('%Y-%m', o.order_date) AS order_month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id != 'UNKNOWN'
    GROUP BY o.customer_id, order_month
),
categorized AS (
    SELECT
        customer_id,
        order_month,
        revenue,
        CASE
            WHEN revenue > 10000 THEN 'High'
            WHEN revenue >= 5000 THEN 'Medium'
            ELSE 'Low'
        END AS revenue_category
    FROM monthly_revenue
)
SELECT
    order_month,
    revenue_category,
    COUNT(*) AS num_customers
FROM categorized
GROUP BY order_month, revenue_category
ORDER BY order_month, revenue_category;

"""

customer_segments = pd.read_sql(query10, conn)

customer_segments

,order_month,revenue_category,num_customers
0,2024-01,Low,2
1,2024-01,Medium,2
2,2024-02,High,1
3,2024-02,Low,2
4,2024-02,Medium,1
...,...,...,...
79,2026-10,Low,4
80,2026-11,Low,2
81,2026-12,High,1
82,2026-12,Low,4


Query 11: NTILE() Segmentation


In [68]:
query11 = """WITH customer_value AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS total_value
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id != 'UNKNOWN'
    GROUP BY o.customer_id
)
SELECT
    customer_id,
    ROUND(total_value, 2) AS total_value,
    NTILE(4) OVER (ORDER BY total_value DESC) AS quartile,
    CASE NTILE(4) OVER (ORDER BY total_value DESC)
        WHEN 1 THEN 'Platinum'
        WHEN 2 THEN 'Gold'
        WHEN 3 THEN 'Silver'
        WHEN 4 THEN 'Bronze'
    END AS quartile_label
FROM customer_value
ORDER BY quartile, total_value DESC;
"""

customer_quartile = pd.read_sql(query11, conn)

customer_quartile

,customer_id,total_value,quartile,quartile_label
0,CUST079,43148.87,1,Platinum
1,CUST021,40435.33,1,Platinum
2,CUST340,28429.96,1,Platinum
3,CUST422,27843.18,1,Platinum
4,CUST121,26699.83,1,Platinum
...,...,...,...,...
208,CUST011,-1216.62,4,Bronze
209,CUST244,-3294.70,4,Bronze
210,CUST299,-4140.67,4,Bronze
211,CUST134,-4840.58,4,Bronze


Query 12: Year-over-Year Comparison

In [69]:
query12 = """WITH monthly AS (
    SELECT
        CAST(strftime('%Y', o.order_date) AS INTEGER) AS year,
        CAST(strftime('%m', o.order_date) AS INTEGER) AS month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY year, month
)
SELECT
    curr.year,
    curr.month,
    ROUND(curr.revenue, 2) AS revenue,
    ROUND(prev.revenue, 2) AS prev_year_revenue,
    CASE
        WHEN prev.revenue IS NULL OR prev.revenue = 0 THEN NULL
        ELSE ROUND((curr.revenue - prev.revenue) * 100.0 / prev.revenue, 2)
    END AS yoy_growth_percent
FROM monthly curr
LEFT JOIN monthly prev
    ON prev.year = curr.year - 1 AND prev.month = curr.month
ORDER BY curr.year, curr.month;
"""
yoy = pd.read_sql(query12, conn)

yoy

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2024,1,23343.55,NaN,NaN
1,2024,2,24589.41,NaN,NaN
2,2024,5,8346.23,NaN,NaN
3,2024,6,22368.97,NaN,NaN
4,2024,7,59415.72,NaN,NaN
5,2024,8,24330.71,NaN,NaN
6,2024,9,24765.67,NaN,NaN
7,2024,10,56538.87,NaN,NaN
8,2024,11,90460.01,NaN,NaN
9,2024,12,80628.19,NaN,NaN


13. First/Last Value Analysis

In [70]:
query13 = """WITH customer_purchases AS (
    SELECT
        o.customer_id,
        o.order_date,
        p.category,
        FIRST_VALUE(p.category) OVER (
            PARTITION BY o.customer_id ORDER BY o.order_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS first_category,
        LAST_VALUE(p.category) OVER (
            PARTITION BY o.customer_id ORDER BY o.order_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS last_category
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.customer_id != 'UNKNOWN'
)
SELECT DISTINCT
    customer_id,
    first_category,
    last_category,
    CASE WHEN first_category != last_category THEN 'Yes' ELSE 'No' END AS category_shift
FROM customer_purchases
ORDER BY customer_id;
"""

category_shift = pd.read_sql(query13, conn)

category_shift

,customer_id,first_category,last_category,category_shift
0,CUST004,Books,Books,No
1,CUST010,Clothing,Clothing,No
2,CUST011,Books,Clothing,Yes
3,CUST014,Electronics,Clothing,Yes
4,CUST016,Clothing,Clothing,No
...,...,...,...,...
208,CUST488,Clothing,Home,Yes
209,CUST492,Books,Books,No
210,CUST494,Electronics,Clothing,Yes
211,CUST497,Clothing,Clothing,No


14. Cumulative Distribution 

In [71]:
query14 = """WITH customer_revenue AS (
    SELECT
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.customer_id != 'UNKNOWN'
    GROUP BY o.customer_id
),
ranked AS (
    SELECT
        customer_id,
        revenue,
        SUM(revenue) OVER (ORDER BY revenue DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_revenue,
        SUM(revenue) OVER () AS total_revenue
    FROM customer_revenue
)
SELECT
    customer_id,
    ROUND(revenue, 2) AS revenue,
    ROUND(cumulative_revenue, 2) AS cumulative_revenue,
    ROUND(cumulative_revenue * 100.0 / total_revenue, 2) AS cumulative_percent
FROM ranked
ORDER BY revenue DESC;
"""

cumulative = pd.read_sql(query14, conn)

cumulative


,customer_id,revenue,cumulative_revenue,cumulative_percent
0,CUST079,43148.87,43148.87,2.59
1,CUST021,40435.33,83584.21,5.02
2,CUST340,28429.96,112014.17,6.73
3,CUST422,27843.18,139857.35,8.40
4,CUST121,26699.83,166557.18,10.00
...,...,...,...,...
208,CUST011,-1216.62,1682180.94,101.04
209,CUST244,-3294.70,1678886.23,100.84
210,CUST299,-4140.67,1674745.56,100.59
211,CUST134,-4840.58,1669904.98,100.30


15. Complex CTE: Cohort Analysis

In [72]:
query15 = """WITH cohorts AS (
    SELECT
        customer_id,
        strftime('%Y-%m', registration_date) AS cohort_month
    FROM customers
),
customer_orders AS (
    SELECT DISTINCT
        c.customer_id,
        c.cohort_month,
        strftime('%Y-%m', o.order_date) AS order_month,
        -- how many calendar months after registration this order happened
        (CAST(strftime('%Y', o.order_date) AS INTEGER) - CAST(substr(c.cohort_month,1,4) AS INTEGER)) * 12
        + (CAST(strftime('%m', o.order_date) AS INTEGER) - CAST(substr(c.cohort_month,6,2) AS INTEGER)) AS month_number
    FROM cohorts c
    JOIN orders o ON o.customer_id = c.customer_id
),
cohort_sizes AS (
    SELECT cohort_month, COUNT(*) AS cohort_size
    FROM cohorts
    GROUP BY cohort_month
)
SELECT
    co.cohort_month,
    cs.cohort_size,
    co.month_number,
    COUNT(DISTINCT co.customer_id) AS customers_ordered,
    ROUND(COUNT(DISTINCT co.customer_id) * 100.0 / cs.cohort_size, 1) AS retention_rate_percent
FROM customer_orders co
JOIN cohort_sizes cs ON cs.cohort_month = co.cohort_month
WHERE co.month_number BETWEEN 0 AND 3
GROUP BY co.cohort_month, co.month_number
ORDER BY co.cohort_month, co.month_number;
"""

cohort_analysis = pd.read_sql(query15, conn)

cohort_analysis

,cohort_month,cohort_size,month_number,customers_ordered,retention_rate_percent
0,2024-04,11,1,1,9.1
1,2024-08,7,3,1,14.3
2,2024-09,7,0,1,14.3
3,2024-09,7,3,1,14.3
4,2024-10,8,0,1,12.5
5,2024-12,10,0,1,10.0
6,2025-01,6,2,1,16.7
7,2025-02,6,0,1,16.7
8,2025-03,13,1,2,15.4
9,2025-03,13,3,2,15.4


Query 16: Self Join

In [73]:
query16 = """SELECT
    p1.product_name AS product_a,
    p2.product_name AS product_b,
    COUNT(*) AS times_bought_together
FROM order_items oi1
JOIN order_items oi2
    ON oi1.order_id = oi2.order_id
    AND oi1.product_id < oi2.product_id   -- "<" avoids duplicates (A-B / B-A) and self-pairs
JOIN products p1 ON p1.product_id = oi1.product_id
JOIN products p2 ON p2.product_id = oi2.product_id
GROUP BY p1.product_name, p2.product_name
ORDER BY times_bought_together DESC
LIMIT 20;
"""

bought_together = pd.read_sql(query16, conn)

bought_together

,product_a,product_b,times_bought_together
0,Above Jacket,Challenge History,1
1,Above Jacket,Score Chair,1
2,Accept Science,Collection Camera,1
3,Action Science,Lawyer Chair,1
4,Action Science,Receive Jacket,1
5,Activity Lamp,Sure Laptop,1
6,Admit Jacket,Lawyer Chair,1
7,Age History,Professional Jacket,1
8,Allow Headphones,Ready Shirt,1
9,Although Shirt,Strategy Headphones,1


## Part 4: Python + SQL Integration

In [77]:
import sqlite3
conn = sqlite3.connect("ecommerce.db")
cursor = conn.cursor()

report_type = input("Enter report type (daily/weekly/monthly): ").lower()

start_date = input("Enter start date (YYYY-MM-DD): ")
end_date = input("Enter end date (YYYY-MM-DD): ")


summary_query = """
SELECT COUNT(DISTINCT o.order_id) AS total_orders,
ROUND(SUM(oi.quantity *  oi.unit_price * (1 - oi.discount_percent / 100.0)),
    2) AS total_revenue,
COUNT(DISTINCT o.customer_id) AS unique_customers
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
WHERE DATE(o.order_date)
BETWEEN ? AND ?;
"""
cursor.execute(summary_query, (start_date, end_date))

summary = cursor.fetchone()


top_products_query = """
SELECT p.product_name,ROUND(
SUM(
oi.quantity *
oi.unit_price *
(1 - oi.discount_percent / 100.0)
),2) AS revenue
FROM order_items oi
JOIN orders o
ON oi.order_id = o.order_id
JOIN products p ON oi.product_id = p.product_id WHERE DATE(o.order_date)
BETWEEN ? AND ?
GROUP BY p.product_name
ORDER BY revenue DESC
LIMIT 3;
"""

cursor.execute(top_products_query, (start_date, end_date))

top_products = cursor.fetchall()


comparison_query = """
SELECT ROUND(
SUM(
oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)
),2)
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
WHERE DATE(o.order_date)
BETWEEN DATE(?, '-1 month')
AND DATE(?, '-1 month');
"""

cursor.execute(comparison_query, (start_date, end_date))
previous_revenue = cursor.fetchone()[0]
current_revenue = summary[1]
if previous_revenue is None or previous_revenue == 0:
    change = "N/A"
else:
    change = round(
        ((current_revenue - previous_revenue) / previous_revenue) * 100,
        2
    )
print("\n")
print("=" * 50)
print("SALES SUMMARY REPORT")
print("=" * 50)

print(f"Report Type       : {report_type.title()}")
print(f"Date Range        : {start_date} to {end_date}")

print("\nSummary")

print(f"Total Orders      : {summary[0]}")
print(f"Total Revenue     : {summary[1]}")
print(f"Unique Customers  : {summary[2]}")

print("\nTop 3 Products")

for product in top_products:

    print(f"{product[0]}  -->  {product[1]}")

print("\nRevenue Change")

print(f"{change}%")

conn.close()



SALES SUMMARY REPORT
Report Type       : Daily
Date Range        : 2023-01-01 to 2024-01-01

Summary
Total Orders      : 0
Total Revenue     : None
Unique Customers  : 0

Top 3 Products

Revenue Change
N/A%


## Part 5: Edge Case Handling

Test Case 1: Invalid order_id

What happens when order_items has an order_id not present in orders?

In [78]:
def test_invalid_order_id():

    test_df = order_items_df.copy()

    test_df.loc[0, "order_id"] = "ORD999999"

    invalid_orders = test_df[
        ~test_df["order_id"].isin(orders_df["order_id"])
    ]

    if len(invalid_orders) > 0:
        print("PASS: Invalid order_id detected.")
    else:
        print("FAIL: Invalid order_id was not detected.")

In [79]:
test_invalid_order_id()

PASS: Invalid order_id detected.


Test Case 2: Discount Greater than 100%


What happens when discount_percent > 100?

In [80]:
def test_invalid_discount():

    test_df = order_items_df.copy()

    test_df.loc[0, "discount_percent"] = 150

    invalid_discount = test_df[
        test_df["discount_percent"] > 100
    ]

    if len(invalid_discount) > 0:
        print("PASS: Invalid discount detected.")
    else:
        print("FAIL: Discount validation failed.")

In [81]:
test_invalid_discount()

PASS: Invalid discount detected.


Test Case 3: Quantity = 0

What happens when quantity is 0?

In [82]:
def test_zero_quantity():

    test_df = order_items_df.copy()

    test_df.loc[0, "quantity"] = 0

    zero_qty = test_df[
        test_df["quantity"] == 0
    ]

    if len(zero_qty) > 0:
        print("PASS: Zero quantity detected.")
    else:
        print("FAIL: Zero quantity validation failed.")

In [83]:
test_zero_quantity()

PASS: Zero quantity detected.


Test Case 4: Future Order Date

What happens when order_date is in the future?

In [84]:
def test_future_order_date():
    from datetime import datetime, timedelta
 
    orders = pd.DataFrame({
        "order_id": [1, 2],
        "order_date": [datetime.now() - timedelta(days=5),
                        datetime.now() + timedelta(days=30)]  
    })
 
    
    now = datetime.now()
    orders["is_future_date"] = orders["order_date"] > now
 
    future_orders = orders[orders["is_future_date"]]
    assert len(future_orders) == 1
    assert future_orders.iloc[0]["order_id"] == 2
    print("TEST 4 PASSED: future order_date values are correctly detected")
 

In [85]:
 test_future_order_date()

TEST 4 PASSED: future order_date values are correctly detected
